# Data Exploration Notebook

This notebook provides tools for exploring documents and understanding the data before processing.

In [ ]:
# Import required libraries
import os
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_colwidth', 100)
plt.style.use('seaborn-v0_8-whitegrid')

print('Libraries loaded successfully!')

## 1. Load and Inspect Documents

In [ ]:
# Load documents from data directory
data_dir = project_root / 'data' / 'raw'

# List available documents
documents = list(data_dir.glob('**/*'))
print(f'Found {len(documents)} items in data/raw/')

for doc in documents[:10]:
    print(f'  - {doc.name}')

In [ ]:
# Import document processing utilities
from backend.core.document_processing import PDFParser, DocumentChunker, MetadataExtractor

# Initialize parsers
pdf_parser = PDFParser()
chunker = DocumentChunker(chunk_size=512, chunk_overlap=50)
metadata_extractor = MetadataExtractor()

## 2. Document Statistics

In [ ]:
# Analyze document statistics
def analyze_text(text: str) -> dict:
    """Compute basic text statistics."""
    words = text.split()
    sentences = text.split('.')
    
    return {
        'char_count': len(text),
        'word_count': len(words),
        'sentence_count': len(sentences),
        'avg_word_length': np.mean([len(w) for w in words]) if words else 0,
        'avg_sentence_length': np.mean([len(s.split()) for s in sentences]) if sentences else 0,
    }

# Example usage
sample_text = "This is a sample document. It contains multiple sentences. We analyze its structure."
stats = analyze_text(sample_text)
print('Sample text statistics:')
for key, value in stats.items():
    print(f'  {key}: {value:.2f}' if isinstance(value, float) else f'  {key}: {value}')

## 3. Chunk Analysis

In [ ]:
# Analyze chunking results
def visualize_chunks(text: str, chunk_size: int = 512, overlap: int = 50):
    """Visualize how text is chunked."""
    chunker = DocumentChunker(chunk_size=chunk_size, chunk_overlap=overlap)
    chunks = chunker.chunk(text, 'test-doc')
    
    chunk_lengths = [len(c.text) for c in chunks]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Chunk length distribution
    axes[0].hist(chunk_lengths, bins=20, edgecolor='black')
    axes[0].set_xlabel('Chunk Length (chars)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Chunk Length Distribution')
    
    # Chunk positions
    positions = [(c.start_index, c.end_index) for c in chunks]
    for i, (start, end) in enumerate(positions):
        axes[1].barh(i, end - start, left=start, height=0.8)
    axes[1].set_xlabel('Character Position')
    axes[1].set_ylabel('Chunk Index')
    axes[1].set_title('Chunk Positions in Document')
    
    plt.tight_layout()
    plt.show()
    
    return chunks

# Example
sample_long_text = ' '.join(['This is sentence number ' + str(i) + '.' for i in range(100)])
# chunks = visualize_chunks(sample_long_text)

## 4. Word Frequency Analysis

In [ ]:
from collections import Counter
import re

def word_frequency_analysis(text: str, top_n: int = 20):
    """Analyze word frequencies in text."""
    # Tokenize and clean
    words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
    
    # Remove stopwords (simple list)
    stopwords = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'in', 'on', 'at', 'to', 'for', 'of', 'and', 'or', 'it', 'this', 'that'}
    words = [w for w in words if w not in stopwords and len(w) > 2]
    
    # Count frequencies
    freq = Counter(words)
    top_words = freq.most_common(top_n)
    
    # Visualize
    if top_words:
        words_list, counts = zip(*top_words)
        
        plt.figure(figsize=(10, 6))
        plt.barh(range(len(words_list)), counts)
        plt.yticks(range(len(words_list)), words_list)
        plt.xlabel('Frequency')
        plt.title(f'Top {top_n} Words')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()
    
    return freq

# Example usage
# freq = word_frequency_analysis(sample_long_text)

## 5. Export Analysis Results

In [ ]:
# Save analysis results
def export_analysis(results: dict, output_path: str):
    """Export analysis results to JSON."""
    import json
    
    with open(output_path, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    
    print(f'Results exported to {output_path}')

# Example
# export_analysis(stats, 'data/exports/analysis_results.json')